# Lab 4 — Orchestrator-Workers: let the LLM split the work (§19.1.4)

The **Cogito Learn** teaching assistant receives 3 papers, and a different task each day

```
goal ──▶ 🧭 orchestrator (LLM writes the subtask plan)
             ├──▶ 👷 worker T1 ─┐
             ├──▶ 👷 worker T2 ─┼──▶ 📝 synthesizer ──▶ one answer
             └──▶ 👷 worker …  ─┘
```

**Unlike parallelization (Lab 3):** the split comes from the model, not hard-coded in the code

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.1.4 (p.370) — CC BY-SA 4.0
> All 3 abstracts are simulated data for teaching; every model output comes from a real run

## 0. Setup

You need Python 3.10+ (or Colab) and an **OpenRouter API key** — setup instructions are in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- from Lab 1: set up client + ask_llm ----
import os, json
from openai import OpenAI

def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"enabled": False}}

def ask_llm(prompt, temperature=0.7, schema=None):
    kw = {"response_format": {"type": "json_schema", "json_schema": {"name": "out", "strict": True, "schema": schema}}} if schema else {}
    resp = client.chat.completions.create(model=MODEL, extra_body=EXTRA, temperature=temperature,
                                          messages=[{"role": "user", "content": prompt}], **kw)
    text = resp.choices[0].message.content
    return json.loads(text) if schema else text

## 1. Inputs and workers

3 simulated papers (P1–P3) and 3 kinds of worker that use **different prompts**

In [ ]:
PAPERS = {  # simulated data for teaching (fictional abstracts)
    "P1": ("TutorBot: An LLM Tutor for Introductory Programming. We deployed an LLM-based tutor in a first-year "
           "Python course with 412 students for one semester. Students who used the tutor at least weekly scored "
           "11% higher on the final exam than a matched control group. The tutor gave hints instead of full solutions."),
    "P2": ("Grading at Scale: Automatic Feedback on Code Assignments. We built a pipeline that runs unit tests and "
           "asks an LLM to explain failures to students. On 3,800 submissions, teaching assistants agreed with the "
           "LLM's feedback in 87% of cases, and grading time per assignment dropped from 9 to 2 minutes."),
    "P3": ("The Hint Trap: Over-reliance on AI Assistants. In a controlled study with 96 students, those given "
           "unlimited AI hints solved 30% more practice problems but scored 14% lower on a closed-book test two "
           "weeks later than students with no AI access. Limiting hints to three per problem removed the gap."),
}

In [ ]:
FACTS_ONLY = ("Copy every number exactly, with the same meaning as in the source. "
              "Add no claim, cause, advice or adjective that is not in the source.")

WORKERS = {
    "summarize": "You summarize one research abstract for students in 2-3 sentences. Keep the key numbers.",
    "extract":   "You extract facts. Return a short bullet list of only what the task asks for. No commentary.",
    "compare":   "You compare papers. In 3-4 sentences, say where they agree and where they conflict. Name papers by id.",
}

## 2. Orchestrator: the LLM writes the subtask plan

The output is JSON (structured output): `{"subtasks": [{"id", "worker", "paper_ids", "instruction"}, …]}`

In [ ]:
SUBTASK = {"type": "object", "additionalProperties": False,
           "required": ["id", "worker", "paper_ids", "instruction"],
           "properties": {"id": {"type": "string"},
                          "worker": {"type": "string", "enum": list(WORKERS)},
                          "paper_ids": {"type": "array", "minItems": 1, "items": {"type": "string", "enum": list(PAPERS)}},
                          "instruction": {"type": "string"}}}

PLAN_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["subtasks"],
               "properties": {"subtasks": {"type": "array", "items": SUBTASK, "minItems": 1}}}

In [ ]:
def plan(goal):
    papers = "\n".join(f"{pid}: {text}" for pid, text in PAPERS.items())
    prompt = f"""You are the orchestrator of a teaching assistant. Break the GOAL into subtasks for workers.
GOAL: {goal}
AVAILABLE PAPERS:
{papers}
WORKER TYPES:
- summarize: overview of ONE paper
- extract: pull the specific facts a question needs from ONE paper
- compare: contrast 2+ papers
RULES:
- Return 1 to 6 subtasks. Use only the subtasks this goal really needs.
- Only use papers that are relevant to the goal.
- Each subtask uses one worker type and lists the paper ids the worker may read.
- summarize and extract read exactly ONE paper: make a separate subtask for each paper.
- Do NOT add a final merge/report subtask; a synthesizer merges the results afterwards.
- ids are T1, T2, ..."""
    return ask_llm(prompt, temperature=0, schema=PLAN_SCHEMA)["subtasks"]

In [ ]:
def show_plan(tasks):
    for t in tasks:
        print(f"  {t['id']}  {t['worker']:<9}  {','.join(t['paper_ids']):<8}  {t['instruction'][:70]}")

In [ ]:
GOAL1 = "Summarize these 3 papers into one overview for students."
tasks1 = plan(GOAL1)
print(f"🧭 PLAN: {len(tasks1)} subtasks")
show_plan(tasks1)

## 3. Workers: do every subtask

Each worker sees **only the papers the orchestrator listed** · the subtasks don't depend on each other → they can run at the same time, as in Lab 3

In [ ]:
def run_worker(task):
    papers = "\n\n".join(f"[{pid}] {PAPERS[pid]}" for pid in task["paper_ids"])
    prompt = f"{WORKERS[task['worker']]} {FACTS_ONLY}\n\nTASK: {task['instruction']}\n\nPAPERS:\n{papers}"
    return ask_llm(prompt, temperature=0.3)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor() as ex:
    results1 = list(ex.map(run_worker, tasks1))

for t, r in zip(tasks1, results1):
    print(f"👷 {t['id']} {t['worker']} {t['paper_ids']}\n{r.strip()[:300]}\n")

## 4. Synthesizer: merge the results into one answer

In [ ]:
def synthesize(goal, tasks, results):
    parts = "\n\n".join(f"[{t['worker']} result, source papers: {', '.join(t['paper_ids'])}]\n{r}"
                        for t, r in zip(tasks, results))
    prompt = (f"GOAL: {goal}\n\nWORKER RESULTS:\n{parts}\n\n"
              "Merge these results into ONE coherent answer for students, in Thai, keeping technical terms "
              "in English, at most 150 words. Use only facts from the results and cite papers as [P1], [P2], [P3]. "
              + FACTS_ONLY)
    return ask_llm(prompt, temperature=0.3)

In [ ]:
print(synthesize(GOAL1, tasks1, results1))

## 5. New goal → new plan (same code)

Combine the three steps into one function, then try it with a student's question

In [ ]:
def orchestrate(goal):
    tasks = plan(goal)
    print(f"🧭 PLAN: {len(tasks)} subtasks"); show_plan(tasks)
    with ThreadPoolExecutor() as ex:
        results = list(ex.map(run_worker, tasks))
    return tasks, synthesize(goal, tasks, results)

In [ ]:
GOAL2 = "A student asks: should I use AI hints while doing practice problems? Answer with evidence."
tasks2, answer2 = orchestrate(GOAL2)
print("\n📝", answer2)

In [ ]:
from collections import Counter

for name, ts in [("GOAL1", tasks1), ("GOAL2", tasks2)]:
    kinds = Counter(t["worker"] for t in ts)
    used = sorted({pid for t in ts for pid in t["paper_ids"]})
    print(f"{name}: {len(ts)} subtasks  {dict(kinds)}  papers={used}")

## Summary

| Part | In the code |
|---|---|
| Orchestrator | `plan()` — one LLM call that writes the plan as JSON following `PLAN_SCHEMA` |
| Workers | `run_worker()` — different prompts per `WORKERS`, reads only the papers it is given · runs in parallel |
| Synthesizer | `synthesize()` — merges the results into one answer, citing [P1]–[P3] |
| Still a workflow | the order plan → workers → synthesize is fixed in the code: no loop, no replanning |

**Try next:** add a 4th worker `"quiz"` (writes 1 review question from a paper) to `WORKERS`,
then try `orchestrate("Make 3 review questions for students from these papers.")` — does the orchestrator pick the new worker on its own?